**Part 2 · LangChain** · step 9 of 25  
[← 08 Conversation history](08_conversation_history.ipynb) · [🏠 Roadmap](../README.md) · [10 First agent →](10_first_agent.ipynb)

# 09 · Tools & Tool Calling

Models can *request* to call tools that fetch data, search the web or run code.
A tool is a pairing of:
1. **A schema** — name, description and argument definitions (a JSON schema).
2. **A function** (or coroutine) that actually executes.

⚠️ The model never runs the tool itself. It returns a **tool call** (name + args);
*your code* (or an agent) executes it and sends the result back.

In [6]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

True

In [7]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4o")

## 1. Create a tool with `@tool`
The decorator reads the function name, type hints and docstring to build the schema.

In [8]:
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    print("I am being called")
    return f"The weather in {location} is sunny"


# Inspect the schema the LLM will see
print("name       :", get_weather.name)
print("description:", get_weather.description)
print("args       :", get_weather.args)



data = get_weather.invoke({"location":"motiharu"})
# print(data)

name       : get_weather
description: Get the weather at a location.
args       : {'location': {'title': 'Location', 'type': 'string'}}
I am being called


## 2. Bind tools to the model
`bind_tools` sends the tool schemas along with every request. The model can now answer with a tool call.

In [9]:
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")
print("content   :", repr(response.content))  # usually empty when the model calls a tool
for tool_call in response.tool_calls:
    print("Tool :", tool_call["name"])
    print("Args :", tool_call["args"])
    print("ID   :", tool_call["id"])

content   : ''
Tool : get_weather
Args : {'location': 'Boston'}
ID   : call_k9kJ2BRzzpYuMOF3mGSuMdZT


## 3. The tool-execution loop (what an agent does for you)

1. Model generates tool calls.
2. **You** execute each tool → a `ToolMessage`.
3. Send everything back → model writes the final answer.

Passing the whole `tool_call` dict to `tool.invoke(...)` returns a ready-made `ToolMessage`
with the correct `tool_call_id`.

In [10]:
# Step 1: the model decides which tools to call
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: execute every requested tool and collect the results
for tool_call in ai_msg.tool_calls:
    print(tool_call)
    tool_result = get_weather.invoke(tool_call)  # -> ToolMessage
    messages.append(tool_result)

# Step 3: send the results back so the model can write the final response
final_response = model_with_tools.invoke(messages)
print(final_response)

{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_EEm4LAa9ZDwAdbieFYdqh9j7', 'type': 'tool_call'}
I am being called
content='The weather in Boston is sunny.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 78, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4f763adedd', 'id': 'chatcmpl-EXZ0S4nxukB8wY7cve49dXaayxToc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a127bd-8b9f-77c0-a218-0ffa2175d6da-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 78, 'output_tokens': 8, 'total_tokens': 86, 'input

## Revision notes
- `@tool` → schema from **name + type hints + docstring**.
- `model.bind_tools([...])` → model *can* emit `response.tool_calls`.
- Manual loop: AIMessage(tool_calls) → ToolMessage(s) → final AIMessage.
- `create_agent` (next notebook, 10) and LangGraph's `ToolNode` (notebook 18) automate this loop.

---
✅ **Done with 09 Tools & tool calling.** Next: [10 First agent →](10_first_agent.ipynb)  
[🏠 Roadmap](../README.md)